# Ewaluacja RAGa

Jednym z narzędzi do ewaluacji systemów RAG i LLMów w ogólności jest biblioteka `deepeval`

https://docs.confident-ai.com/docs/getting-started

## Quickstart

Aby dokonać ewaluacji działania systemu opartego o LLM, np. RAG używamy klasy `LLMTestCase`.

Test case'y bazują na następujących parametrach:
- `input` (prompt użytkownika)
- `actual_output` (rzeczywista odpowiedź modelu)
- `expected_output` (oczekiwana odpowiedź modelu)
- `retrieval_context` (kontekst pozyskany z bazy wektorowej)

Parametrów tych jest więcej, ale w kontekście metryk dla RAGa będziemy używać wyłącznie powyższych.

In [ ]:
from dotenv import load_dotenv

from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig

In [ ]:
load_dotenv()

Dokumentacja `LLMTestCase`: https://docs.confident-ai.com/docs/evaluation-test-cases

In [ ]:
test_case = LLMTestCase(
    input="What if these shoes don't fit?",
    actual_output="We offer a 30-day full refund at no extra cost.",
    expected_output="You're eligible for a 30 day refund at no extra cost.",
    retrieval_context=["All customers are eligible for a 30 day full refund at no extra cost."]
)

In [ ]:
answer_relevancy_metric = AnswerRelevancyMetric(threshold=0.5, model="gpt-4o")

In [ ]:
import os
os.environ["DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"] = "100"

In [ ]:
result = evaluate([test_case], [answer_relevancy_metric])

In [ ]:
result

In [ ]:
result.test_results

In [ ]:
result.test_results[0].metrics_data[0]

In [ ]:
result.test_results[0].metrics_data[0].score

## Metryki

W bibliotece `deepeval` występuje 5 metryk, które mogą być pomocne do zastosowań związanych z RAG:

**retrieval**
- `ContextualPrecisionMetric`
- `ContextualRecallMetric`
- `ContextualRelevancyMetric`

**generation**
- `AnswerRelevancyMetric`
- `FaithfulnessMetric`


Oprócz tego istnieje również możliwość tworzenia customowych metryk za pomocą klasy `GEval`.

### Retrieval

#### `ContextualPrecisionMetric`

Ocenia, czy reranker w retrieverze umieszcza bardziej istotne fragmenty w kontekście wyszukiwania wyżej niż fragmenty mniej istotne.

Innymi słowy sprawdzamy, czy kolejność pozyskanych dokumentów jest optymalna w kontekście zadanego pytania.

https://docs.confident-ai.com/docs/metrics-contextual-precision

---

**Zestawiamy:** kolejność elementów w `retrieval_context` i `model_input`

In [ ]:
from deepeval.metrics import ContextualPrecisionMetric


model_input = "What if these shoes don't fit?"
actual_output = "It doesn't matter."
expected_output = "It doesn't matter as well."

retrieval_context_1 = ["All customers are eligible for a 30 day full refund at no extra cost.",
                       "For each pair of shoes bought, you get a 10% discount for the next purchase."]

retrieval_context_2 = ["For each pair of shoes bought, you get a 10% discount for the next purchase.",
                       "All customers are eligible for a 30 day full refund at no extra cost."]

retrieval_context_3 = ["At our shop we sell shoes",
                       "For each pair of shoes bought, you get a 10% discount for the next purchase."]


metric = ContextualPrecisionMetric(
    threshold=0.7,
    model="gpt-4o"
)

test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

Aby policzyć wartość metryki należy wywołać funkcję `evaluate()` przekazując do niej m.in. poniższe argumenty:
- listę przypadków testowych – może być to lista jednoelementowa
- listę metryk – również może być jednoelementowa
- `print_results` (opcjonalnie) – do kontroli *verbosity*
- `run_async` (opcjonalnie) – wywołanie asynchroniczne przyspieszy działanie kodu, ale nie zapewnia nam określonej kolejności wyników

In [ ]:
result = evaluate(
    test_cases, [metric],
    async_config=AsyncConfig(run_async=True),
    display_config=DisplayConfig(print_results=True)
)

In [ ]:
print("Model input:", result.test_results[0].input, "\n=================================================================\n\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nContext:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

#### `ContextualRecallMetric`

Ocenia, czy retriever potrafi poprawnie dobrać właściwe informacje do tego co jest w oczekiwanej odpowiedzi.

https://docs.confident-ai.com/docs/metrics-contextual-recall

---

**Zestawiamy:** `retrieval_context` oraz `expected_output`

In [ ]:
from deepeval.metrics import ContextualRecallMetric


model_input = "It doesn't matter"
actual_output = "It doesn't matter as well"
expected_output = "You are eligible for a 30 day full refund at no extra cost."

retrieval_context_1 = ["All customers are eligible for a 30 day full refund at no extra cost.", "At our shop we sell shoes"]
retrieval_context_2 = ["All customers are eligible for a 30 day full refund at no extra cost."]
retrieval_context_3 = ["For each pair of shoes bought, you get a 10% discount for the next purchase.", "At our shop we sell shoes"]


metric = ContextualRecallMetric(
    threshold=0.7,
    model="gpt-4o"
)


test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Expected output:", result.test_results[0].expected_output, "\n=================================================================\n\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nContext:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

#### `ContextualRelevancyMetric`

Ocenia jaki odsetek fragmentów zwróconych przez retriever dotyczy tego, czego dotyczy prompt wejściowy.

https://docs.confident-ai.com/docs/metrics-contextual-relevancy

---

**Zestawiamy:** proporcję relewantnych tekstów w `retrieval_context` w odniesieniu do `model_input`

In [ ]:
from deepeval.metrics import ContextualRelevancyMetric


model_input = "What if these shoes don't fit?"
actual_output = "It doesn't matter"
expected_output = "It doesn't matter as well"

retrieval_context_1 = ["All customers are eligible for a 30 day full refund at no extra cost.", "At our shop we sell shoes"]
retrieval_context_2 = ["All customers are eligible for a 30 day full refund at no extra cost."]
retrieval_context_3 = ["For each pair of shoes bought, you get a 10% discount for the next purchase.", "At our shop we sell shoes"]


metric = ContextualRelevancyMetric(
    threshold=0.7,
    model="gpt-4o"
)



test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                  async_config=AsyncConfig(run_async=False),
                  display_config=DisplayConfig(print_results=False))

In [ ]:
print("Model input:", result.test_results[0].input, "\n=================================================================\n\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nContext:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

### Generation

#### `AnswerRelevancyMetric`

Określa na ile odpowiedź modelu jest adekwatna w odniesieniu do inputu użytkownika.

https://docs.confident-ai.com/docs/metrics-answer-relevancy

---

**Zestawiamy:** `model_input` i `actual_output`.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric


model_input = "What if these shoes don't fit?"

actual_output_1 = "We offer a 30-day full refund at no extra cost."
actual_output_2 = "If the shoes don't fit, ..."
actual_output_3 = "At our shop we sell shoes."

metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o"
)


test_cases = []
for actual_output in [actual_output_1, actual_output_2, actual_output_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Model input:", result.test_results[0].input, "\n=================================================================\n\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nActual output:\n{test_result.actual_output}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

#### `FaithfulnessMetric`

Określa czy odpowiedź modelu odnosi się w do pozyskanego z bazy kotenkstu.

https://docs.confident-ai.com/docs/metrics-faithfulness

---

**Zestawiamy:** `actual_output` i `retrieval_context`.

In [ ]:
from deepeval.metrics import FaithfulnessMetric


model_input = "It doesn't matter"
actual_output = "We offer a 30-day full refund at no extra cost."

retrieval_context_1 = ["All customers are eligible for a 30 day full refund at no extra cost."]
retrieval_context_2 = ["At our shop we sell shoes."]
retrieval_context_3 = ["It's not possible to offer any refund to the customer."]

metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o"
)


test_cases = []
for retrieval_context in [retrieval_context_1, retrieval_context_2, retrieval_context_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        retrieval_context=retrieval_context
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Model output:", result.test_results[0].actual_output, "\n=================================================================\n\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nRetrieval context:\n{test_result.retrieval_context}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

### Podsumowanie metryk

| Nazwa metryki      | Zestawienie |
|--------------------|---|
| `ContextualPrecision`   | kolejność `retrieval_context` i `model_input` |
| `ContextualRecall`      | `retrieval_context` i `expected_output` |
| `ContextualRelevancy`   | `retrieval_context` i `model_input` |
| `AnswerRelevancy`       | `actual_output` i `model_input`|
| `Faithfulness`           | `actual_output` i `retrieval_context` |


### Customowe metryki

#### `GEval`

Użyjemy `GEval` aby stworzyć własną metrykę o nazwie "Correctness", która porówna oczekiwaną oraz rzeczywistą odpowiedź modelu.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

In [ ]:
metric = GEval(
        name="Correctness",
        criteria="""Determine if the 'actual output' is correct based on the 'expected output'.
                    Consider general sense of the answer and lack of contradictions. The answer doesn't need to be exactly the same""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.7,
        model="gpt-4o"
    )


model_input = "It doesn't matter"
actual_output = "We offer a 30-day full refund at no extra cost."
expected_output_1 = "We offer a 15-day full refund at no extra cost."
expected_output_2 = "We offer a 30-day total refund with any additional payments."
expected_output_3 = "We don't offer any refund"



test_cases = []
for expected_output in [expected_output_1, expected_output_2, expected_output_3]:
    test_case = LLMTestCase(
        input=model_input,
        actual_output=actual_output,
        expected_output=expected_output
    )

    test_cases.append(test_case)

In [ ]:
result = evaluate(test_cases, [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
print("Actual output:", result.test_results[0].actual_output, "\n=================================================================\n\n")

for test_result in result.test_results:
    metric_data = test_result.metrics_data[0]
    
    print(f"score: {metric_data.score}", f"threshold: {metric_data.threshold}", f"success: {metric_data.success}",
          f"------------\nExpected output::\n{test_result.expected_output}\n------------", f"Reason:\n{metric_data.reason}", sep="\n", end="\n\n======\n\n")

## Ewaluacja RAGa – use case

Use case na przykładzie `knowledge_database`

In [ ]:
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai.chat_models import ChatOpenAI

In [ ]:
client = QdrantClient(url="http://localhost:6333")

In [ ]:
embeddings = HuggingFaceEmbeddings(model="sdadas/st-polish-paraphrase-from-distilroberta")

In [ ]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="knowledge_database",
    embedding=embeddings,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
template = """Retrieved documents:
---------
{context}

-----
End of retrieved documents
============

Answer the following question: {input}.

=============

Important information: You can answer only based on the context from the retrieved documents.
If you don't have any information regarding the question in the context, say 'I don't know'"""


prompt_template = PromptTemplate.from_template(template)

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

---

In [ ]:
model_input = "Powiedz mi kto czym jeździ"
response = retrieval_chain.invoke({"input": model_input})

In [ ]:
expected_output = "Andrzej jeździ samochodem, Bartek rowerem a Czarek tramwajem."
retrieval_context = [doc.page_content for doc in response["context"]]

In [ ]:
retrieval_context

In [ ]:
actual_output = response["answer"]
actual_output

### Metryki

#### `ContextualPrecisionMetric`

**Zestawiamy:** kolejność `retrieval_context` oraz `model_input`

In [ ]:
metric = ContextualPrecisionMetric(
    threshold=0.7,
    model="gpt-4o"
)


test_case = LLMTestCase(
    input=model_input,
    actual_output=actual_output,
    expected_output=expected_output,
    retrieval_context=retrieval_context
)

In [ ]:
result = evaluate([test_case], [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
metric_data = result.test_results[0].metrics_data[0]    

print(f"score: {metric_data.score}")
print(f"threshold: {metric_data.threshold}")
print(f"success: {metric_data.success}")
print(f"\nModel input:\n{result.test_results[0].input}")
print(f"\nContext:\n{result.test_results[0].retrieval_context}")
print(f"\nReason:\n{metric_data.reason}")

#### `ContextualRecallMetric`

**Zestawiamy:** `retrieval_context` oraz `expected_output`

In [ ]:
metric = ContextualRecallMetric(
    threshold=0.7,
    model="gpt-4o"
)

In [ ]:
result = evaluate([test_case], [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
metric_data = result.test_results[0].metrics_data[0]    

print(f"score: {metric_data.score}")
print(f"threshold: {metric_data.threshold}")
print(f"success: {metric_data.success}")
print(f"\nExpected output:\n{result.test_results[0].expected_output}")
print(f"\nContext:\n{result.test_results[0].retrieval_context}")
print(f"\nReason:\n{metric_data.reason}")

#### `ContextualRelevancyMetric`

**Zestawiamy:** `retrieval_context` oraz `model_input`

In [ ]:
metric = ContextualRelevancyMetric(
    threshold=0.7,
    model="gpt-4o"
)

In [ ]:
result = evaluate([test_case], [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
metric_data = result.test_results[0].metrics_data[0]    

print(f"score: {metric_data.score}")
print(f"threshold: {metric_data.threshold}")
print(f"success: {metric_data.success}")
print(f"\nModel input:\n{result.test_results[0].input}")
print(f"\nContext:\n{result.test_results[0].retrieval_context}")
print(f"\nReason:\n{metric_data.reason}")

#### `AnswerRelevancyMetric`

**Zestawiamy:** `actual_output` oraz `model_input`

In [ ]:
metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o"
)

In [ ]:
result = evaluate([test_case], [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
metric_data = result.test_results[0].metrics_data[0]    

print(f"score: {metric_data.score}")
print(f"threshold: {metric_data.threshold}")
print(f"success: {metric_data.success}")
print(f"\nModel input:\n{result.test_results[0].input}")
print(f"\nModel output:\n{result.test_results[0].actual_output}")
print(f"\nReason:\n{metric_data.reason}")

#### `FaithfulnessMetric`

**Zestawiamy:** `retrieval_context` oraz `actual_output`

In [ ]:
metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o"
)

In [ ]:
result = evaluate([test_case], [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
metric_data = result.test_results[0].metrics_data[0]    

print(f"score: {metric_data.score}")
print(f"threshold: {metric_data.threshold}")
print(f"success: {metric_data.success}")
print(f"\nContext:\n{result.test_results[0].retrieval_context}")
print(f"\nModel output:\n{result.test_results[0].actual_output}")
print(f"\nReason:\n{metric_data.reason}")

#### `Correctness` (`GEval`)

**Zestawiamy:** `actual_output` oraz `expected_output`

In [ ]:
metric = GEval(
        name="Correctness",
        criteria="""Determine if the 'actual output' is correct based on the 'expected output'.
                    Consider general sense of the answer and lack of contradictions. The answer doesn't need to be exactly the same""",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
        threshold=0.7,
        model="gpt-4o"
    )

In [ ]:
result = evaluate([test_case], [metric],
                 async_config=AsyncConfig(run_async=False),
                 display_config=DisplayConfig(print_results=False))

In [ ]:
metric_data = result.test_results[0].metrics_data[0]    

print(f"score: {metric_data.score}")
print(f"threshold: {metric_data.threshold}")
print(f"success: {metric_data.success}")
print(f"\nExpected output:\n{result.test_results[0].expected_output}")
print(f"\nModel output:\n{result.test_results[0].actual_output}")
print(f"\nReason:\n{metric_data.reason}")